In [ ]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import CountVectorizer

from gensim.models import Word2Vec, FastText

ModuleNotFoundError: No module named 'pandas'

In [ ]:
df = pd.read_csv("../data/cleaned_imdb_dataset.csv")

print("Dataset Loaded Successfully!")
print("Dataset Shape:", df.shape)

print(df.head())
print(df.columns)

In [ ]:
columns = {col.lower(): col for col in df.columns}

possible_columns = ["cleaned_review", "cleaned_text", "review", "text"]

text_column = None

for col in possible_columns:
    if col in columns:
        text_column = columns[col]
        break

if text_column is None:
    raise ValueError("Text column not found. Check the dataset columns.")

texts = df[text_column].dropna().astype(str)

texts = texts[texts.str.strip() != ""]

print("Total Documents:", len(texts))
print("Sample Review:")
print(texts.iloc[0])

In [ ]:
bow_vectorizer = CountVectorizer()

bow_matrix = bow_vectorizer.fit_transform(texts)

print("Bag of Words Matrix Shape:", bow_matrix.shape)

total_elements = bow_matrix.shape[0] * bow_matrix.shape[1]

zero_elements = total_elements - bow_matrix.nnz

sparsity = (zero_elements / total_elements) * 100

print("Sparsity:", sparsity, "%")

print("\nVocabulary Size:", len(bow_vectorizer.vocabulary_))

In [ ]:
print("First 20 Vocabulary Words:")

print(bow_vectorizer.get_feature_names_out()[:20])

print("\nFirst 5 Rows of BoW Matrix:")

print(bow_matrix[:5].toarray())

In [ ]:
unigram_vectorizer = CountVectorizer(ngram_range=(1, 1))

unigram_matrix = unigram_vectorizer.fit_transform(texts)


bigram_vectorizer = CountVectorizer(ngram_range=(2, 2))

bigram_matrix = bigram_vectorizer.fit_transform(texts)


trigram_vectorizer = CountVectorizer(ngram_range=(3, 3))

trigram_matrix = trigram_vectorizer.fit_transform(texts)


print("Unigram Shape:", unigram_matrix.shape)

print("Bigram Shape:", bigram_matrix.shape)

print("Trigram Shape:", trigram_matrix.shape)

In [ ]:
def calculate_sparsity(matrix):

    total = matrix.shape[0] * matrix.shape[1]

    zeros = total - matrix.nnz

    return (zeros / total) * 100


ngram_results = pd.DataFrame({

    "N-gram": ["Unigram", "Bigram", "Trigram"],

    "Vocabulary Size": [
        unigram_matrix.shape[1],
        bigram_matrix.shape[1],
        trigram_matrix.shape[1]
    ],

    "Sparsity (%)": [
        calculate_sparsity(unigram_matrix),
        calculate_sparsity(bigram_matrix),
        calculate_sparsity(trigram_matrix)
    ]

})

print(ngram_results)

In [ ]:
print("Unigram Examples:")

print(unigram_vectorizer.get_feature_names_out()[:20])


print("\nBigram Examples:")

print(bigram_vectorizer.get_feature_names_out()[:20])


print("\nTrigram Examples:")

print(trigram_vectorizer.get_feature_names_out()[:20])

In [ ]:
tokenized_reviews = []

for review in texts:

    words = review.split()

    if len(words) > 0:

        tokenized_reviews.append(words)


print("Total Tokenized Reviews:", len(tokenized_reviews))

print("\nSample Tokenized Review:")

print(tokenized_reviews[0])

In [ ]:
cbow_model = Word2Vec(

    sentences=tokenized_reviews,

    vector_size=100,

    window=5,

    min_count=2,

    workers=4,

    sg=0,

    epochs=10,

    seed=42

)

print("Word2Vec CBOW Model Trained Successfully!")

In [ ]:
print("Vector Size:", cbow_model.vector_size)

print("Window Size:", cbow_model.window)

print("Minimum Count:", cbow_model.min_count)

print("Epochs:", cbow_model.epochs)

print("Vocabulary Size:", len(cbow_model.wv))

In [ ]:
word = "movie"

if word in cbow_model.wv:

    print("Vector Representation of:", word)

    print(cbow_model.wv[word])

else:

    print(word, "is not present in the vocabulary.")

In [ ]:
fasttext_model = FastText(

    sentences=tokenized_reviews,

    vector_size=100,

    window=5,

    min_count=2,

    workers=4,

    sg=0,

    epochs=10,

    min_n=3,

    max_n=6,

    seed=42

)

print("fastText Model Trained Successfully!")

In [ ]:
print("Vector Size:", fasttext_model.vector_size)

print("Window Size:", fasttext_model.window)

print("Minimum Count:", fasttext_model.min_count)

print("Epochs:", fasttext_model.epochs)

print("Vocabulary Size:", len(fasttext_model.wv))

In [ ]:
known_word = "amazing"

if known_word not in fasttext_model.wv:

    known_word = fasttext_model.wv.index_to_key[0]


misspelled_word = known_word[:-1]

print("Original Word:", known_word)

print("Misspelled Word:", misspelled_word)

In [ ]:
try:

    print(cbow_model.wv[misspelled_word])

except KeyError:

    print("Word2Vec cannot generate a vector for this OOV word.")

In [ ]:
fasttext_vector = fasttext_model.wv.get_vector(misspelled_word)

print("fastText generated a vector successfully!")

print("Vector:")

print(fasttext_vector)

In [ ]:
similar_words = fasttext_model.wv.most_similar(

    positive=[fasttext_vector],

    topn=5

)

print("Similar Words:")

for word, score in similar_words:

    print(word, ":", score)